
# Attention + Bidirectional Encoder：較接近真實實作的 NumPy 版本

本 Notebook 是前一版的延伸版，目標是把原本「數學推導對應程式」的範例，往更接近真實機器翻譯實作推進。

本版新增重點：

1. 使用 `pandas` 讀取 `Data/fra.txt`。
2. 英文 source vocabulary 與法文 target vocabulary 分開。
3. 使用 `<PAD>`、`<SOS>`、`<EOS>`、`<UNK>`。
4. 加入 padding 與 source attention mask。
5. 使用 mini-batch 梯度累積。
6. 使用 Adam optimizer。
7. 加入 teacher forcing ratio。
8. 加入 greedy decoding 與 beam search decoding。
9. 加入 token accuracy、exact match 與 loss 曲線。
10. 加入 attention heatmap。

限制仍維持教學取向：

- 模型、forward、backward、optimizer 都使用 NumPy 手寫。
- 不使用 PyTorch、TensorFlow、Keras。
- 為了讓程式碼可讀，mini-batch 採「batch 內逐筆 forward/backward，再平均梯度」的寫法。這比完全向量化慢，但更容易看懂 BPTT。


## 程式碼風格說明

本 Notebook 採用以下原則：

> 核心數學與模型流程避免過度語法糖；一般資料處理、列印、檢查與輔助函式可使用 Python 慣用寫法，以可讀性為優先。

因此，RNN time step、attention score、softmax backward、BPTT、梯度累積與參數更新等核心數學流程會保留較展開的寫法，方便對照數學式。資料處理、詞彙表建立、padding、列印與解碼輔助函式則可使用 `enumerate`、list comprehension、dictionary comprehension、`.get()` 等常見 Python 寫法，避免重複樣板干擾閱讀。


In [ ]:

import math
import random
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(7)
random.seed(7)



## 1. 用 pandas 讀取英法資料

資料路徑設定為：

```text
Data/fra.txt
```

請把資料放在 Notebook 同層的 `Data` 資料夾中。

`fra.txt` 通常是 tab-separated text，也就是每列用 tab 分隔英文與法文：

```text
Go.    Va !
Run!   Cours !
```

有些版本的檔案可能有第三欄來源資訊，所以這裡只讀前兩欄。


In [ ]:

DATA_PATH = "Data/fra.txt"

def load_fra_txt_with_pandas(path):
    # fra.txt 是 tab-separated text，可用 pandas.read_csv 讀取。
    # header=None 表示檔案本身沒有欄位名稱。
    # usecols=[0, 1] 只取英文與法文欄位。
    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        usecols=[0, 1],
        names=["english", "french"],
        dtype=str,
        encoding="utf-8"
    )

    df = df.dropna(subset=["english", "french"]).reset_index(drop=True)
    return df

raw_df = load_fra_txt_with_pandas(DATA_PATH)
print("原始資料筆數:", len(raw_df))
display(raw_df.head())



## 2. 文字正規化與斷詞

真實 NLP 專案通常需要更完整的 tokenizer，例如 BPE、WordPiece、SentencePiece。

這裡仍使用簡單規則斷詞，原因是本 Notebook 的重點是讓讀者看懂 attention 與 BPTT，不把複雜度放在 tokenizer。


In [ ]:

def normalize_sentence(text):
    text = str(text)
    text = text.lower().strip()
    text = text.replace("’", "'")

    # 把標點切成獨立 token，例如 "hello!" -> "hello !"
    text = re.sub(r"([.!?,';:])", r" \1 ", text)

    # 保留英文、法文重音字母、數字與常見標點。
    text = re.sub(r"[^a-zA-ZÀ-ÖØ-öø-ÿ0-9.!?,';:]+", " ", text)

    # 合併多餘空白。
    text = re.sub(r"\s+", " ", text).strip()

    return text

def tokenize(text):
    text = normalize_sentence(text)

    if text == "":
        return []

    return text.split(" ")

def build_pairs_from_dataframe(df, max_pairs=2000, max_src_len=7, max_tgt_len=9):
    pairs = []
    seen_english = set()

    for row_index in range(len(df)):
        src_tokens = tokenize(df.loc[row_index, "english"])
        tgt_tokens = tokenize(df.loc[row_index, "french"])

        if len(src_tokens) == 0 or len(tgt_tokens) == 0:
            continue

        if len(src_tokens) > max_src_len:
            continue

        if len(tgt_tokens) > max_tgt_len:
            continue

        # 同一句英文可能有多個法文翻譯。教學版先保留第一個，避免一對多答案干擾觀察。
        src_key = " ".join(src_tokens)

        if src_key in seen_english:
            continue

        seen_english.add(src_key)
        pairs.append((src_tokens, tgt_tokens))

        if len(pairs) >= max_pairs:
            break

    return pairs

pairs = build_pairs_from_dataframe(
    raw_df,
    max_pairs=2000,
    max_src_len=7,
    max_tgt_len=9
)

print("可用短句 pair 數:", len(pairs))
for i in range(10):
    print(pairs[i])



## 3. 分開建立 source vocabulary 與 target vocabulary

前一版為了簡化，英文與法文共用同一個 vocabulary。

真實翻譯模型通常會分開：

- `src_token_to_id`：英文輸入詞彙表。
- `tgt_token_to_id`：法文輸出詞彙表。
- `E_src`：英文 embedding。
- `E_tgt`：法文 embedding。
- 輸出層只對 target vocabulary 做分類。

這樣比較符合 encoder-decoder 翻譯模型的實務設計。


In [ ]:

PAD = "<PAD>"
SOS = "<SOS>"
EOS = "<EOS>"
UNK = "<UNK>"

def build_vocab(token_lists, max_vocab_size):
    counter = Counter()

    for tokens in token_lists:
        counter.update(tokens)

    special_tokens = [PAD, SOS, EOS, UNK]
    vocab = special_tokens + [
        token
        for token, count in counter.most_common(max_vocab_size - len(special_tokens))
    ]

    token_to_id = {token: index for index, token in enumerate(vocab)}
    id_to_token = {index: token for index, token in enumerate(vocab)}

    return vocab, token_to_id, id_to_token

src_token_lists = [src_tokens for src_tokens, tgt_tokens in pairs]
tgt_token_lists = [tgt_tokens for src_tokens, tgt_tokens in pairs]

src_vocab, src_token_to_id, src_id_to_token = build_vocab(src_token_lists, max_vocab_size=250)
tgt_vocab, tgt_token_to_id, tgt_id_to_token = build_vocab(tgt_token_lists, max_vocab_size=300)

SRC_PAD_ID = src_token_to_id[PAD]
SRC_UNK_ID = src_token_to_id[UNK]

TGT_PAD_ID = tgt_token_to_id[PAD]
TGT_SOS_ID = tgt_token_to_id[SOS]
TGT_EOS_ID = tgt_token_to_id[EOS]
TGT_UNK_ID = tgt_token_to_id[UNK]

print("英文 source vocabulary 大小:", len(src_vocab))
print("法文 target vocabulary 大小:", len(tgt_vocab))
print("source vocab 前 20 個:", src_vocab[:20])
print("target vocab 前 20 個:", tgt_vocab[:20])


In [ ]:

def encode_tokens(tokens, token_to_id, unk_id):
    return [token_to_id.get(token, unk_id) for token in tokens]

def decode_target_ids(ids):
    tokens = []

    for token_id in ids:
        token = tgt_id_to_token[int(token_id)]

        if token == EOS:
            break

        if token != PAD and token != SOS:
            tokens.append(token)

    return tokens

def detokenize(tokens):
    text = " ".join(tokens)

    text = text.replace(" .", ".")
    text = text.replace(" !", "!")
    text = text.replace(" ?", "?")
    text = text.replace(" ,", ",")
    text = text.replace(" ;", ";")
    text = text.replace(" :", ":")
    text = text.replace(" ' ", "'")
    text = text.replace("' ", "'")

    return text

def make_encoded_dataset(pairs):
    dataset = []

    for src_tokens, tgt_tokens in pairs:
        src_ids = encode_tokens(src_tokens, src_token_to_id, SRC_UNK_ID)
        tgt_ids = encode_tokens(tgt_tokens, tgt_token_to_id, TGT_UNK_ID)
        tgt_ids.append(TGT_EOS_ID)

        example = {
            "src_tokens": src_tokens,
            "tgt_tokens": tgt_tokens,
            "src_ids": src_ids,
            "tgt_ids": tgt_ids
        }

        dataset.append(example)

    return dataset

encoded_dataset = make_encoded_dataset(pairs)
random.shuffle(encoded_dataset)

split_index = int(len(encoded_dataset) * 0.85)
train_examples = encoded_dataset[:split_index]
valid_examples = encoded_dataset[split_index:]

print("訓練資料筆數:", len(train_examples))
print("驗證資料筆數:", len(valid_examples))
print("範例:", train_examples[0])



## 4. Padding 與 mini-batch

真實訓練通常不會逐筆送進模型，而是組成 mini-batch。

因為每句長度不同，所以需要 padding：

```text
I am cold <PAD> <PAD>
I love you
```

同時要建立 mask：

```text
1 1 1 0 0
1 1 1
```

在 attention 裡，padding 位置不能被模型注意到，因此 softmax 前要把 padding 的 score 設成極小值。


In [ ]:

def pad_sequence(ids, max_len, pad_id):
    padding_length = max_len - len(ids)
    padded = list(ids) + [pad_id] * padding_length
    mask = [1.0] * len(ids) + [0.0] * padding_length
    return padded, mask

def make_batches(examples, batch_size, shuffle=True):
    data = list(examples)

    if shuffle:
        random.shuffle(data)

    batches = []

    for start in range(0, len(data), batch_size):
        batch_examples = data[start:start + batch_size]

        max_src_len = 0
        max_tgt_len = 0

        for ex in batch_examples:
            max_src_len = max(max_src_len, len(ex["src_ids"]))
            max_tgt_len = max(max_tgt_len, len(ex["tgt_ids"]))

        src_batch = []
        src_mask_batch = []
        tgt_batch = []
        tgt_mask_batch = []

        for ex in batch_examples:
            src_padded, src_mask = pad_sequence(ex["src_ids"], max_src_len, SRC_PAD_ID)
            tgt_padded, tgt_mask = pad_sequence(ex["tgt_ids"], max_tgt_len, TGT_PAD_ID)

            src_batch.append(src_padded)
            src_mask_batch.append(src_mask)
            tgt_batch.append(tgt_padded)
            tgt_mask_batch.append(tgt_mask)

        batch = {
            "examples": batch_examples,
            "src_ids": np.array(src_batch, dtype=np.int64),
            "src_mask": np.array(src_mask_batch, dtype=np.float64),
            "tgt_ids": np.array(tgt_batch, dtype=np.int64),
            "tgt_mask": np.array(tgt_mask_batch, dtype=np.float64)
        }

        batches.append(batch)

    return batches

demo_batch = make_batches(train_examples[:4], batch_size=4, shuffle=False)[0]
print("src_ids shape:", demo_batch["src_ids"].shape)
print("src_mask shape:", demo_batch["src_mask"].shape)
print("tgt_ids shape:", demo_batch["tgt_ids"].shape)
print("tgt_mask shape:", demo_batch["tgt_mask"].shape)
print("src ids:")
print(demo_batch["src_ids"])
print("src mask:")
print(demo_batch["src_mask"])



## 5. 模型參數初始化

模型架構：

1. Source embedding：英文 token id → 向量。
2. Bidirectional RNN Encoder。
3. Additive Attention。
4. Decoder RNN。
5. Target output projection：decoder 狀態 → 法文 vocabulary 機率。

和前一版相比，本版把 `E_src` 與 `E_tgt` 分開。


In [ ]:

def init_matrix(rows, cols):
    limit = math.sqrt(6.0 / (rows + cols))
    return np.random.uniform(-limit, limit, (rows, cols))

def init_translation_params(src_vocab_size, tgt_vocab_size, src_embed_dim, tgt_embed_dim, enc_hidden_dim, dec_hidden_dim, attn_dim):
    params = {}

    params["E_src"] = init_matrix(src_embed_dim, src_vocab_size)
    params["E_tgt"] = init_matrix(tgt_embed_dim, tgt_vocab_size)

    params["Wxh_f"] = init_matrix(enc_hidden_dim, src_embed_dim)
    params["Whh_f"] = init_matrix(enc_hidden_dim, enc_hidden_dim)
    params["bh_f"] = np.zeros((enc_hidden_dim, 1))

    params["Wxh_b"] = init_matrix(enc_hidden_dim, src_embed_dim)
    params["Whh_b"] = init_matrix(enc_hidden_dim, enc_hidden_dim)
    params["bh_b"] = np.zeros((enc_hidden_dim, 1))

    enc_concat_dim = enc_hidden_dim * 2

    params["W_init"] = init_matrix(dec_hidden_dim, enc_concat_dim)
    params["b_init"] = np.zeros((dec_hidden_dim, 1))

    params["Wa"] = init_matrix(attn_dim, dec_hidden_dim)
    params["Ua"] = init_matrix(attn_dim, enc_concat_dim)
    params["va"] = init_matrix(attn_dim, 1)

    params["Wy"] = init_matrix(dec_hidden_dim, tgt_embed_dim)
    params["Ws"] = init_matrix(dec_hidden_dim, dec_hidden_dim)
    params["Wc"] = init_matrix(dec_hidden_dim, enc_concat_dim)
    params["bs"] = np.zeros((dec_hidden_dim, 1))

    params["Wo"] = init_matrix(tgt_vocab_size, dec_hidden_dim + enc_concat_dim)
    params["bo"] = np.zeros((tgt_vocab_size, 1))

    return params

def zero_grads_like(params):
    return {key: np.zeros_like(value) for key, value in params.items()}

def one_hot(token_id, vocab_size):
    vector = np.zeros((vocab_size, 1))
    vector[int(token_id), 0] = 1.0
    return vector

def embed_src(params, token_id):
    oh = one_hot(token_id, params["E_src"].shape[1])
    emb = np.dot(params["E_src"], oh)
    return emb, oh

def embed_tgt(params, token_id):
    oh = one_hot(token_id, params["E_tgt"].shape[1])
    emb = np.dot(params["E_tgt"], oh)
    return emb, oh



## 6. Masked softmax

在 attention 中，若 source 句子有 padding，padding 位置不應該被關注。

做法：

$$
e_i =
\begin{cases}
e_i, & \text{mask}_i = 1 \\
-10^9, & \text{mask}_i = 0
\end{cases}
$$

再做 softmax。


In [ ]:

def softmax(logits):
    shifted = logits - np.max(logits)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values)

def masked_softmax(logits, mask):
    # logits shape: (T, 1)
    # mask shape: (T,)
    masked_logits = np.copy(logits)

    for i in range(len(mask)):
        if mask[i] == 0.0:
            masked_logits[i, 0] = -1e9

    shifted = masked_logits - np.max(masked_logits)
    exp_values = np.exp(shifted)

    for i in range(len(mask)):
        if mask[i] == 0.0:
            exp_values[i, 0] = 0.0

    total = np.sum(exp_values)

    if total <= 0.0:
        # 理論上不應發生，除非整句都是 PAD。
        return np.ones_like(logits) / logits.shape[0]

    return exp_values / total

def cross_entropy_from_probs(probs, target_id):
    eps = 1e-12
    return -math.log(float(probs[int(target_id), 0]) + eps)



## 7. Forward pass

本函式處理單筆已 padding 的資料。

雖然資料有 padding，但會透過：

- `src_mask` 控制 attention。
- `tgt_mask` 控制 loss 是否計入 `<PAD>`。

為了維持 BPTT 清楚，encoder 仍會跑完整個 padded source 長度；attention 會遮掉 padding 位置。


In [ ]:

def forward_example(params, src_ids, src_mask, tgt_ids, tgt_mask, teacher_forcing_ratio=1.0):
    cache = {}

    Tx = len(src_ids)
    Ty = len(tgt_ids)

    src_embeddings = []
    src_one_hots = []

    for i in range(Tx):
        emb, oh = embed_src(params, src_ids[i])
        src_embeddings.append(emb)
        src_one_hots.append(oh)

    # Encoder forward direction
    h_f = []
    prev_h = np.zeros((params["Whh_f"].shape[0], 1))

    for i in range(Tx):
        a = np.dot(params["Wxh_f"], src_embeddings[i]) + np.dot(params["Whh_f"], prev_h) + params["bh_f"]
        h_candidate = np.tanh(a)

        # 若是 PAD，讓 hidden state 沿用上一個狀態，降低 padding 對 encoder 的干擾。
        if src_mask[i] == 1.0:
            h = h_candidate
        else:
            h = prev_h

        h_f.append(h)
        prev_h = h

    # Encoder backward direction
    h_b = [None] * Tx
    next_h = np.zeros((params["Whh_b"].shape[0], 1))

    for i in range(Tx - 1, -1, -1):
        a = np.dot(params["Wxh_b"], src_embeddings[i]) + np.dot(params["Whh_b"], next_h) + params["bh_b"]
        h_candidate = np.tanh(a)

        if src_mask[i] == 1.0:
            h = h_candidate
        else:
            h = next_h

        h_b[i] = h
        next_h = h

    h_enc = []

    for i in range(Tx):
        h_enc.append(np.vstack((h_f[i], h_b[i])))

    # 找出最後一個非 PAD 的 source 位置。
    last_real_index = 0
    for i in range(Tx):
        if src_mask[i] == 1.0:
            last_real_index = i

    first_real_index = 0

    enc_summary = np.vstack((h_f[last_real_index], h_b[first_real_index]))
    a_init = np.dot(params["W_init"], enc_summary) + params["b_init"]
    s_prev = np.tanh(a_init)

    decoder_input_ids = []
    decoder_input_ids.append(TGT_SOS_ID)

    # Teacher forcing：訓練時通常餵真實前一個 token；也可按比例改餵模型預測。
    # 本 forward 為了讓 backward 清楚，當 teacher_forcing_ratio < 1 時只影響下一步輸入，不改變 loss 定義。
    previous_pred_id = TGT_SOS_ID

    for t in range(1, Ty):
        use_teacher = random.random() < teacher_forcing_ratio

        if use_teacher:
            decoder_input_ids.append(int(tgt_ids[t - 1]))
        else:
            decoder_input_ids.append(previous_pred_id)

        # previous_pred_id 會在主迴圈中更新。這裡先放 placeholder，主迴圈會直接使用 decoder_input_ids[t]。
        previous_pred_id = int(tgt_ids[t - 1])

    s_list = []
    c_list = []
    alpha_list = []
    z_list = []
    output_concat_list = []
    probs_list = []
    decoder_input_embeddings = []
    decoder_input_one_hots = []

    loss = 0.0
    valid_steps = 0.0

    for t in range(Ty):
        y_emb, y_oh = embed_tgt(params, decoder_input_ids[t])
        decoder_input_embeddings.append(y_emb)
        decoder_input_one_hots.append(y_oh)

        e_vec = np.zeros((Tx, 1))
        z_values = []

        for i in range(Tx):
            z = np.tanh(np.dot(params["Wa"], s_prev) + np.dot(params["Ua"], h_enc[i]))
            e = np.dot(params["va"].T, z)
            z_values.append(z)
            e_vec[i, 0] = e[0, 0]

        alpha = masked_softmax(e_vec, src_mask)

        c = np.zeros_like(h_enc[0])

        for i in range(Tx):
            c = c + alpha[i, 0] * h_enc[i]

        a_dec = (
            np.dot(params["Wy"], y_emb)
            + np.dot(params["Ws"], s_prev)
            + np.dot(params["Wc"], c)
            + params["bs"]
        )

        s = np.tanh(a_dec)
        output_concat = np.vstack((s, c))
        logits = np.dot(params["Wo"], output_concat) + params["bo"]
        probs = softmax(logits)

        if tgt_mask[t] == 1.0:
            loss = loss + cross_entropy_from_probs(probs, int(tgt_ids[t]))
            valid_steps = valid_steps + 1.0

        previous_pred_id = int(np.argmax(probs[:, 0]))

        s_list.append(s)
        c_list.append(c)
        alpha_list.append(alpha)
        z_list.append(z_values)
        output_concat_list.append(output_concat)
        probs_list.append(probs)

        s_prev = s

    if valid_steps > 0.0:
        loss = loss / valid_steps

    cache["src_ids"] = src_ids
    cache["src_mask"] = src_mask
    cache["tgt_ids"] = tgt_ids
    cache["tgt_mask"] = tgt_mask
    cache["Tx"] = Tx
    cache["Ty"] = Ty
    cache["valid_steps"] = valid_steps
    cache["src_embeddings"] = src_embeddings
    cache["src_one_hots"] = src_one_hots
    cache["h_f"] = h_f
    cache["h_b"] = h_b
    cache["h_enc"] = h_enc
    cache["enc_summary"] = enc_summary
    cache["a_init"] = a_init
    cache["last_real_index"] = last_real_index
    cache["first_real_index"] = first_real_index
    cache["decoder_input_ids"] = decoder_input_ids
    cache["decoder_input_embeddings"] = decoder_input_embeddings
    cache["decoder_input_one_hots"] = decoder_input_one_hots
    cache["s_list"] = s_list
    cache["c_list"] = c_list
    cache["alpha_list"] = alpha_list
    cache["z_list"] = z_list
    cache["output_concat_list"] = output_concat_list
    cache["probs_list"] = probs_list

    return loss, cache



## 8. Backward pass

本段仍然是手寫 BPTT。

和前一版主要差異：

1. `E_src` 與 `E_tgt` 分開更新。
2. loss 只對非 padding 的 target token 計算。
3. attention softmax backward 會自然透過 masked softmax 的 alpha，把 padding 權重壓到 0。
4. batch 梯度由多筆 example 的梯度加總後平均。


In [ ]:

def backward_example(params, cache):
    grads = zero_grads_like(params)

    Tx = cache["Tx"]
    Ty = cache["Ty"]

    enc_hidden_dim = params["Whh_f"].shape[0]
    dec_hidden_dim = params["Ws"].shape[0]

    valid_steps = cache["valid_steps"]

    if valid_steps <= 0.0:
        return grads

    dh_enc = []
    for _ in range(Tx):
        dh_enc.append(np.zeros_like(cache["h_enc"][0]))

    dh_f_extra = []
    dh_b_extra = []

    for _ in range(Tx):
        dh_f_extra.append(np.zeros_like(cache["h_f"][0]))
        dh_b_extra.append(np.zeros_like(cache["h_b"][0]))

    ds_from_future = np.zeros((dec_hidden_dim, 1))

    for t in range(Ty - 1, -1, -1):
        if cache["tgt_mask"][t] == 0.0:
            continue

        target_id = int(cache["tgt_ids"][t])

        dlogits = np.copy(cache["probs_list"][t])
        dlogits[target_id, 0] = dlogits[target_id, 0] - 1.0
        dlogits = dlogits / valid_steps

        output_concat = cache["output_concat_list"][t]

        grads["Wo"] = grads["Wo"] + np.dot(dlogits, output_concat.T)
        grads["bo"] = grads["bo"] + dlogits

        doutput_concat = np.dot(params["Wo"].T, dlogits)

        ds = doutput_concat[0:dec_hidden_dim, :]
        dc = doutput_concat[dec_hidden_dim:, :]

        ds = ds + ds_from_future

        s = cache["s_list"][t]
        da_dec = ds * (1.0 - s * s)

        if t == 0:
            s_prev = np.tanh(cache["a_init"])
        else:
            s_prev = cache["s_list"][t - 1]

        y_emb = cache["decoder_input_embeddings"][t]
        c = cache["c_list"][t]

        grads["Wy"] = grads["Wy"] + np.dot(da_dec, y_emb.T)
        grads["Ws"] = grads["Ws"] + np.dot(da_dec, s_prev.T)
        grads["Wc"] = grads["Wc"] + np.dot(da_dec, c.T)
        grads["bs"] = grads["bs"] + da_dec

        ds_prev_total = np.dot(params["Ws"].T, da_dec)
        dc = dc + np.dot(params["Wc"].T, da_dec)

        dy_emb = np.dot(params["Wy"].T, da_dec)
        grads["E_tgt"] = grads["E_tgt"] + np.dot(dy_emb, cache["decoder_input_one_hots"][t].T)

        alpha = cache["alpha_list"][t]
        dalpha = np.zeros((Tx, 1))

        for i in range(Tx):
            dh_enc[i] = dh_enc[i] + alpha[i, 0] * dc
            dalpha[i, 0] = np.dot(dc.T, cache["h_enc"][i])[0, 0]

        # softmax backward。padding 的 alpha 接近 0，因此不會產生有效梯度。
        weighted_sum = float(np.sum(dalpha * alpha))
        de = np.zeros((Tx, 1))

        for i in range(Tx):
            de[i, 0] = alpha[i, 0] * (dalpha[i, 0] - weighted_sum)

        for i in range(Tx):
            z = cache["z_list"][t][i]
            dz = de[i, 0] * params["va"]
            da_attn = dz * (1.0 - z * z)

            grads["va"] = grads["va"] + de[i, 0] * z
            grads["Wa"] = grads["Wa"] + np.dot(da_attn, s_prev.T)
            grads["Ua"] = grads["Ua"] + np.dot(da_attn, cache["h_enc"][i].T)

            ds_prev_total = ds_prev_total + np.dot(params["Wa"].T, da_attn)
            dh_enc[i] = dh_enc[i] + np.dot(params["Ua"].T, da_attn)

        if t > 0:
            ds_from_future = ds_prev_total
        else:
            s_init = np.tanh(cache["a_init"])
            da_init = ds_prev_total * (1.0 - s_init * s_init)

            grads["W_init"] = grads["W_init"] + np.dot(da_init, cache["enc_summary"].T)
            grads["b_init"] = grads["b_init"] + da_init

            denc_summary = np.dot(params["W_init"].T, da_init)

            last_real_index = cache["last_real_index"]
            first_real_index = cache["first_real_index"]

            dh_f_extra[last_real_index] = dh_f_extra[last_real_index] + denc_summary[0:enc_hidden_dim, :]
            dh_b_extra[first_real_index] = dh_b_extra[first_real_index] + denc_summary[enc_hidden_dim:, :]

            ds_from_future = np.zeros_like(ds_from_future)

    for i in range(Tx):
        dh_f_extra[i] = dh_f_extra[i] + dh_enc[i][0:enc_hidden_dim, :]
        dh_b_extra[i] = dh_b_extra[i] + dh_enc[i][enc_hidden_dim:, :]

    # Forward encoder BPTT
    dh_next = np.zeros((enc_hidden_dim, 1))

    for i in range(Tx - 1, -1, -1):
        if cache["src_mask"][i] == 0.0:
            continue

        dh_total = dh_f_extra[i] + dh_next
        h = cache["h_f"][i]
        da = dh_total * (1.0 - h * h)

        x_emb = cache["src_embeddings"][i]

        if i == 0:
            h_prev = np.zeros((enc_hidden_dim, 1))
        else:
            h_prev = cache["h_f"][i - 1]

        grads["Wxh_f"] = grads["Wxh_f"] + np.dot(da, x_emb.T)
        grads["Whh_f"] = grads["Whh_f"] + np.dot(da, h_prev.T)
        grads["bh_f"] = grads["bh_f"] + da

        dx_emb = np.dot(params["Wxh_f"].T, da)
        grads["E_src"] = grads["E_src"] + np.dot(dx_emb, cache["src_one_hots"][i].T)

        dh_next = np.dot(params["Whh_f"].T, da)

    # Backward encoder BPTT
    dh_prev_in_backward_time = np.zeros((enc_hidden_dim, 1))

    for i in range(0, Tx):
        if cache["src_mask"][i] == 0.0:
            continue

        dh_total = dh_b_extra[i] + dh_prev_in_backward_time
        h = cache["h_b"][i]
        da = dh_total * (1.0 - h * h)

        x_emb = cache["src_embeddings"][i]

        if i == Tx - 1:
            h_next = np.zeros((enc_hidden_dim, 1))
        else:
            h_next = cache["h_b"][i + 1]

        grads["Wxh_b"] = grads["Wxh_b"] + np.dot(da, x_emb.T)
        grads["Whh_b"] = grads["Whh_b"] + np.dot(da, h_next.T)
        grads["bh_b"] = grads["bh_b"] + da

        dx_emb = np.dot(params["Wxh_b"].T, da)
        grads["E_src"] = grads["E_src"] + np.dot(dx_emb, cache["src_one_hots"][i].T)

        dh_prev_in_backward_time = np.dot(params["Whh_b"].T, da)

    return grads



## 9. Mini-batch 梯度累積

為了讓程式易讀，這裡不是把整個 batch 完全矩陣化，而是：

1. 一個 batch 裡逐筆 example 計算 forward/backward。
2. 把梯度加總。
3. 除以 batch size。
4. 用 Adam 更新參數。

這在教學上比較清楚，也保留 mini-batch 訓練的概念。


In [ ]:

def add_grads(total_grads, grads):
    for key in total_grads:
        total_grads[key] += grads[key]

def scale_grads(grads, scale):
    for key in grads:
        grads[key] *= scale

def global_grad_norm(grads):
    total = sum(float(np.sum(grad * grad)) for grad in grads.values())
    return math.sqrt(total)

def clip_grads(grads, max_norm):
    norm = global_grad_norm(grads)

    if norm > max_norm:
        scale = max_norm / (norm + 1e-12)

        for key in grads:
            grads[key] = grads[key] * scale

    return norm

def compute_batch_loss_and_grads(params, batch, teacher_forcing_ratio):
    total_grads = zero_grads_like(params)
    total_loss = 0.0
    count = 0

    batch_size = batch["src_ids"].shape[0]

    for row in range(batch_size):
        loss, cache = forward_example(
            params=params,
            src_ids=batch["src_ids"][row],
            src_mask=batch["src_mask"][row],
            tgt_ids=batch["tgt_ids"][row],
            tgt_mask=batch["tgt_mask"][row],
            teacher_forcing_ratio=teacher_forcing_ratio
        )

        grads = backward_example(params, cache)

        add_grads(total_grads, grads)
        total_loss = total_loss + loss
        count = count + 1

    if count > 0:
        scale_grads(total_grads, 1.0 / count)
        total_loss = total_loss / count

    return total_loss, total_grads



## 10. Adam optimizer

SGD 可以學，但容易不穩定。真實深度學習訓練常用 Adam。

Adam 對每個參數維護：

- 一階動量 `m`
- 二階動量 `v`

更新式：

$$
m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t
$$

$$
v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2
$$

$$
\theta \leftarrow \theta - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon}
$$


In [ ]:

def init_adam_state(params):
    return {
        "m": {key: np.zeros_like(value) for key, value in params.items()},
        "v": {key: np.zeros_like(value) for key, value in params.items()},
        "t": 0
    }

def adam_update(params, grads, state, learning_rate=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
    state["t"] = state["t"] + 1
    t = state["t"]

    for key in params:
        g = grads[key]

        state["m"][key] = beta1 * state["m"][key] + (1.0 - beta1) * g
        state["v"][key] = beta2 * state["v"][key] + (1.0 - beta2) * (g * g)

        m_hat = state["m"][key] / (1.0 - beta1 ** t)
        v_hat = state["v"][key] / (1.0 - beta2 ** t)

        params[key] = params[key] - learning_rate * m_hat / (np.sqrt(v_hat) + eps)



## 11. 推論：greedy decoding

Greedy decoding 每一步都選機率最高的 token。

優點是快；缺點是容易卡在局部最佳解。


In [ ]:

def encode_source_sentence(sentence):
    tokens = tokenize(sentence)
    ids = encode_tokens(tokens, src_token_to_id, SRC_UNK_ID)
    mask = np.ones((len(ids),), dtype=np.float64)
    return tokens, np.array(ids, dtype=np.int64), mask

def encode_source_from_ids(src_ids):
    src_mask = np.ones((len(src_ids),), dtype=np.float64)
    return np.array(src_ids, dtype=np.int64), src_mask

def run_encoder(params, src_ids, src_mask):
    Tx = len(src_ids)

    src_embeddings = []

    for i in range(Tx):
        emb, _ = embed_src(params, src_ids[i])
        src_embeddings.append(emb)

    h_f = []
    prev_h = np.zeros((params["Whh_f"].shape[0], 1))

    for i in range(Tx):
        a = np.dot(params["Wxh_f"], src_embeddings[i]) + np.dot(params["Whh_f"], prev_h) + params["bh_f"]
        h = np.tanh(a)

        if src_mask[i] == 0.0:
            h = prev_h

        h_f.append(h)
        prev_h = h

    h_b = [None] * Tx
    next_h = np.zeros((params["Whh_b"].shape[0], 1))

    for i in range(Tx - 1, -1, -1):
        a = np.dot(params["Wxh_b"], src_embeddings[i]) + np.dot(params["Whh_b"], next_h) + params["bh_b"]
        h = np.tanh(a)

        if src_mask[i] == 0.0:
            h = next_h

        h_b[i] = h
        next_h = h

    h_enc = []

    for i in range(Tx):
        h_enc.append(np.vstack((h_f[i], h_b[i])))

    last_real_index = 0

    for i in range(Tx):
        if src_mask[i] == 1.0:
            last_real_index = i

    enc_summary = np.vstack((h_f[last_real_index], h_b[0]))
    s0 = np.tanh(np.dot(params["W_init"], enc_summary) + params["b_init"])

    return h_enc, s0

def decoder_step(params, h_enc, src_mask, s_prev, input_token_id):
    Tx = len(h_enc)

    y_emb, _ = embed_tgt(params, input_token_id)

    e_vec = np.zeros((Tx, 1))

    for i in range(Tx):
        z = np.tanh(np.dot(params["Wa"], s_prev) + np.dot(params["Ua"], h_enc[i]))
        e = np.dot(params["va"].T, z)
        e_vec[i, 0] = e[0, 0]

    alpha = masked_softmax(e_vec, src_mask)

    c = np.zeros_like(h_enc[0])

    for i in range(Tx):
        c = c + alpha[i, 0] * h_enc[i]

    a_dec = (
        np.dot(params["Wy"], y_emb)
        + np.dot(params["Ws"], s_prev)
        + np.dot(params["Wc"], c)
        + params["bs"]
    )

    s = np.tanh(a_dec)
    output_concat = np.vstack((s, c))
    logits = np.dot(params["Wo"], output_concat) + params["bo"]
    probs = softmax(logits)

    return probs, s, alpha

def greedy_decode(params, src_ids, max_len=15):
    src_ids, src_mask = encode_source_from_ids(src_ids)
    h_enc, s_prev = run_encoder(params, src_ids, src_mask)

    input_token_id = TGT_SOS_ID
    output_ids = []
    attentions = []

    for step in range(max_len):
        probs, s, alpha = decoder_step(params, h_enc, src_mask, s_prev, input_token_id)

        next_id = int(np.argmax(probs[:, 0]))

        output_ids.append(next_id)
        attentions.append(alpha[:, 0].copy())

        if next_id == TGT_EOS_ID:
            break

        input_token_id = next_id
        s_prev = s

    return output_ids, attentions

def translate_greedy(params, sentence, max_len=15):
    src_tokens, src_ids, src_mask = encode_source_sentence(sentence)
    output_ids, attentions = greedy_decode(params, src_ids, max_len=max_len)
    output_tokens = decode_target_ids(output_ids)

    return src_tokens, output_tokens, attentions



## 12. Beam search decoding

Beam search 不只保留當下機率最高的一條路，而是保留多條候選。

例如 `beam_width = 3` 時，每一步保留累積 log probability 最高的 3 條序列。

這比較接近真實翻譯系統的 decoding。


In [ ]:

def beam_search_decode(params, src_ids, max_len=15, beam_width=3):
    src_ids, src_mask = encode_source_from_ids(src_ids)
    h_enc, s0 = run_encoder(params, src_ids, src_mask)

    # 每個 beam: (token_ids, score, decoder_state, attentions, ended)
    beams = [
        ([], 0.0, s0, [], False)
    ]

    for step in range(max_len):
        candidates = []

        for token_ids, score, s_prev, attentions, ended in beams:
            if ended:
                candidates.append((token_ids, score, s_prev, attentions, ended))
                continue

            if len(token_ids) == 0:
                input_token_id = TGT_SOS_ID
            else:
                input_token_id = token_ids[-1]

            probs, s_next, alpha = decoder_step(params, h_enc, src_mask, s_prev, input_token_id)

            # 取機率最高的前 beam_width 個 token。
            sorted_ids = np.argsort(-probs[:, 0])

            for k in range(beam_width):
                next_id = int(sorted_ids[k])
                next_prob = float(probs[next_id, 0])
                next_score = score + math.log(next_prob + 1e-12)

                new_token_ids = list(token_ids)
                new_token_ids.append(next_id)

                new_attentions = list(attentions)
                new_attentions.append(alpha[:, 0].copy())

                new_ended = next_id == TGT_EOS_ID

                candidates.append((new_token_ids, next_score, s_next, new_attentions, new_ended))

        # 長度正規化，避免 beam search 過度偏好短句。
        def normalized_score(item):
            token_ids, score, s_state, attentions, ended = item
            length = max(1, len(token_ids))
            return score / length

        candidates.sort(key=normalized_score, reverse=True)
        beams = candidates[:beam_width]

        all_ended = True

        for item in beams:
            if not item[4]:
                all_ended = False

        if all_ended:
            break

    best = beams[0]
    return best[0], best[3]

def translate_beam(params, sentence, max_len=15, beam_width=3):
    src_tokens, src_ids, src_mask = encode_source_sentence(sentence)
    output_ids, attentions = beam_search_decode(params, src_ids, max_len=max_len, beam_width=beam_width)
    output_tokens = decode_target_ids(output_ids)

    return src_tokens, output_tokens, attentions



## 13. 評估指標

本版加入三種評估：

1. validation loss。
2. token accuracy：非 padding target token 的預測正確率。
3. exact match：整句完全相同才算正確。

翻譯任務中 exact match 很嚴格，所以 token accuracy 通常比較能看出早期訓練是否有進步。


In [ ]:

def evaluate_loss(params, examples, batch_size=16):
    batches = make_batches(examples, batch_size=batch_size, shuffle=False)

    total_loss = 0.0
    count = 0

    for batch in batches:
        batch_size_actual = batch["src_ids"].shape[0]

        for row in range(batch_size_actual):
            loss, cache = forward_example(
                params=params,
                src_ids=batch["src_ids"][row],
                src_mask=batch["src_mask"][row],
                tgt_ids=batch["tgt_ids"][row],
                tgt_mask=batch["tgt_mask"][row],
                teacher_forcing_ratio=1.0
            )

            total_loss = total_loss + loss
            count = count + 1

    return total_loss / max(1, count)

def evaluate_decoding(params, examples, max_items=100, max_len=15):
    total_tokens = 0
    correct_tokens = 0
    exact_match = 0
    evaluated = 0

    subset = examples[:max_items]

    for ex in subset:
        pred_ids, attentions = greedy_decode(params, ex["src_ids"], max_len=max_len)
        pred_tokens = decode_target_ids(pred_ids)
        target_tokens = ex["tgt_tokens"]

        compare_len = max(len(pred_tokens), len(target_tokens))

        for i in range(compare_len):
            pred_token = None
            target_token = None

            if i < len(pred_tokens):
                pred_token = pred_tokens[i]

            if i < len(target_tokens):
                target_token = target_tokens[i]

            if pred_token == target_token:
                correct_tokens = correct_tokens + 1

            total_tokens = total_tokens + 1

        if pred_tokens == target_tokens:
            exact_match = exact_match + 1

        evaluated = evaluated + 1

    token_accuracy = correct_tokens / max(1, total_tokens)
    exact_match_accuracy = exact_match / max(1, evaluated)

    return token_accuracy, exact_match_accuracy



## 14. 訓練函式

這裡結合：

- mini-batch
- gradient clipping
- Adam
- teacher forcing ratio decay
- validation loss
- token accuracy
- exact match


In [ ]:

def train_translation_model(
    params,
    train_examples,
    valid_examples,
    epochs=12,
    batch_size=16,
    learning_rate=0.002,
    clip_norm=5.0,
    teacher_forcing_start=1.0,
    teacher_forcing_end=0.7
):
    adam_state = init_adam_state(params)

    history = {
        "train_loss": [],
        "valid_loss": [],
        "token_accuracy": [],
        "exact_match": []
    }

    for epoch in range(epochs):
        if epochs == 1:
            teacher_forcing_ratio = teacher_forcing_end
        else:
            progress = epoch / (epochs - 1)
            teacher_forcing_ratio = teacher_forcing_start + progress * (teacher_forcing_end - teacher_forcing_start)

        batches = make_batches(train_examples, batch_size=batch_size, shuffle=True)

        total_train_loss = 0.0
        total_grad_norm = 0.0
        batch_count = 0

        for batch in batches:
            batch_loss, grads = compute_batch_loss_and_grads(
                params=params,
                batch=batch,
                teacher_forcing_ratio=teacher_forcing_ratio
            )

            grad_norm = clip_grads(grads, clip_norm)
            adam_update(
                params=params,
                grads=grads,
                state=adam_state,
                learning_rate=learning_rate
            )

            total_train_loss = total_train_loss + batch_loss
            total_grad_norm = total_grad_norm + grad_norm
            batch_count = batch_count + 1

        avg_train_loss = total_train_loss / max(1, batch_count)
        valid_loss = evaluate_loss(params, valid_examples, batch_size=batch_size)
        token_acc, exact_acc = evaluate_decoding(params, valid_examples, max_items=100, max_len=15)

        history["train_loss"].append(avg_train_loss)
        history["valid_loss"].append(valid_loss)
        history["token_accuracy"].append(token_acc)
        history["exact_match"].append(exact_acc)

        print(
            "epoch", epoch + 1,
            "| train_loss", round(avg_train_loss, 4),
            "| valid_loss", round(valid_loss, 4),
            "| token_acc", round(token_acc, 4),
            "| exact_match", round(exact_acc, 4),
            "| teacher_forcing", round(teacher_forcing_ratio, 3),
            "| avg_grad_norm", round(total_grad_norm / max(1, batch_count), 4)
        )

    return history



## 15. 開始訓練

這裡的超參數刻意保守，讓一般電腦也能執行。若你想要更好的翻譯結果，可以增加：

- `max_pairs`
- `max_vocab_size`
- `epochs`
- hidden dimension
- beam width

但 NumPy 手寫版本會明顯變慢。


In [ ]:

params = init_translation_params(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    src_embed_dim=32,
    tgt_embed_dim=32,
    enc_hidden_dim=40,
    dec_hidden_dim=48,
    attn_dim=40
)

history = train_translation_model(
    params=params,
    train_examples=train_examples,
    valid_examples=valid_examples,
    epochs=12,
    batch_size=16,
    learning_rate=0.002,
    clip_norm=5.0,
    teacher_forcing_start=1.0,
    teacher_forcing_end=0.75
)



## 16. 訓練曲線


In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], label="train loss")
plt.plot(history["valid_loss"], label="valid loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("Training / Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history["token_accuracy"], label="token accuracy")
plt.plot(history["exact_match"], label="exact match")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()



## 17. 測試翻譯：greedy vs beam search

這裡比較兩種 decoding：

- greedy decoding：每一步選最大機率。
- beam search：保留多條候選路徑。


In [ ]:

manual_tests = [
    "i am cold",
    "i am hungry",
    "i love you",
    "thank you",
    "good luck",
    "how are you",
    "i need water",
    "tom is here",
    "we are ready",
    "please stop"
]

for sentence in manual_tests:
    src_tokens, greedy_tokens, greedy_attn = translate_greedy(params, sentence, max_len=15)
    src_tokens, beam_tokens, beam_attn = translate_beam(params, sentence, max_len=15, beam_width=3)

    print("EN input :", sentence)
    print("tokens   :", src_tokens)
    print("greedy   :", detokenize(greedy_tokens))
    print("beam     :", detokenize(beam_tokens))
    print("-" * 70)



## 18. 驗證集翻譯結果抽樣


In [ ]:

for i in range(15):
    ex = valid_examples[i]

    greedy_ids, greedy_attn = greedy_decode(params, ex["src_ids"], max_len=15)
    beam_ids, beam_attn = beam_search_decode(params, ex["src_ids"], max_len=15, beam_width=3)

    greedy_tokens = decode_target_ids(greedy_ids)
    beam_tokens = decode_target_ids(beam_ids)

    print("EN input :", detokenize(ex["src_tokens"]))
    print("FR target:", detokenize(ex["tgt_tokens"]))
    print("greedy   :", detokenize(greedy_tokens))
    print("beam     :", detokenize(beam_tokens))
    print("-" * 70)



## 19. Attention heatmap

這張圖可以檢查模型在產生每個法文 token 時，大致關注哪些英文 token。

若模型有學到一些對齊關係，常會看到：

- `i` 對應到 `je`
- `am` 對應到 `suis` 或 `ai`
- 名詞或形容詞有局部對應


In [ ]:

example_sentence = "i am cold"

src_tokens, output_tokens, attentions = translate_beam(
    params,
    example_sentence,
    max_len=15,
    beam_width=3
)

attention_matrix = np.array(attentions)

if len(output_tokens) > 0 and attention_matrix.shape[0] > 0:
    plt.figure(figsize=(8, 5))
    plt.imshow(attention_matrix[:len(output_tokens), :], aspect="auto")
    plt.colorbar(label="attention weight")
    plt.xticks(range(len(src_tokens)), src_tokens)
    plt.yticks(range(len(output_tokens)), output_tokens)
    plt.xlabel("English source token")
    plt.ylabel("French predicted token")
    plt.title("Attention Heatmap")
    plt.show()

print("EN input :", example_sentence)
print("FR pred  :", detokenize(output_tokens))



## 20. 這一版和教學版的差異整理

| 項目 | 前一版 | 本版 |
|---|---|---|
| 資料讀取 | Python open | pandas |
| 詞彙表 | 英法共用 | source / target 分開 |
| padding | 幾乎不處理 | 有 padding 與 mask |
| attention | 一般 softmax | masked softmax |
| 訓練 | 逐筆 SGD | mini-batch 梯度累積 |
| optimizer | SGD | Adam |
| decoding | greedy | greedy + beam search |
| 評估 | loss / exact match | loss / token accuracy / exact match |
| 實務接近度 | 教學入門 | 更接近真實 Seq2Seq 實作 |

仍可繼續延伸：

1. 把 batch 內運算完全向量化。
2. 加入 dropout。
3. 加入 layer normalization。
4. 改成 GRU 或 LSTM。
5. 改用 subword tokenizer。
6. 加入 BLEU score。
7. 儲存與載入模型參數。
8. 加入 gradient checking。
9. 改寫成 Transformer encoder-decoder。
